# video-descriptor

Sube un video, extrae el audio con `ffmpeg` y transcribe con WhisperX. Salidas: `JSON`, `TXT`, `SRT` y `VTT`.

Antes de ejecutar: `Runtime > Change runtime type > GPU`.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg
!pip install -q --upgrade-strategy only-if-needed \
  -r https://raw.githubusercontent.com/pballestawork/video-descriptor/main/requirements-colab.txt \
  git+https://github.com/pballestawork/video-descriptor.git

# Colab may ask for a runtime restart after dependency changes.
!python - <<'PY'
import numpy, pandas, whisperx
print('numpy', numpy.__version__)
print('pandas', pandas.__version__)
print('whisperx', getattr(whisperx, '__version__', 'installed'))
PY

## Subida interactiva

Esta ruta es cómoda, pero con vídeos largos puede fallar por navegador, red, sesión o disco temporal. Si falla, usa la celda de Drive/URL de abajo.

In [ ]:
from google.colab import files

uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No se ha subido ningún archivo")

VIDEO_PATH = next(iter(uploaded.keys()))
print(f"Video seleccionado: {VIDEO_PATH}")

## Fallback para vídeos grandes

Si la subida interactiva no aguanta el tamaño del vídeo, sube el archivo a Google Drive y copia una versión local a `/content` antes de ejecutar.

In [ ]:
# Opción Drive. Descomenta y ajusta la ruta si la subida interactiva falla.
# from google.colab import drive
# drive.mount('/content/drive')
# !cp "/content/drive/MyDrive/path/to/video.mp4" "/content/video.mp4"
# VIDEO_PATH = "/content/video.mp4"

# Opción URL descargable. Descomenta y ajusta si tienes una URL directa.
# !wget -O "/content/video.mp4" "https://example.com/video.mp4"
# VIDEO_PATH = "/content/video.mp4"

## Transcripción

Valores por defecto: español (`es`), `large-v3-turbo`, sin diarización.

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path('/content/video_descriptor_outputs')

!python -m video_descriptor transcribe "$VIDEO_PATH" \
  --output-dir "$OUTPUT_DIR" \
  --model large-v3-turbo \
  --language es \
  --batch-size 16 \
  --compute-type float16 \
  --device auto

In [ ]:
import shutil
from google.colab import files

archive = shutil.make_archive('/content/video_descriptor_outputs', 'zip', OUTPUT_DIR)
files.download(archive)